In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from credit_risk import config as C, data as D, features as F, scorecard as S, models as M
from credit_risk import calibration as CAL, business as B, fairness as FAIR, stability as ST
R = lambda s: json.load(open(C.REPORTS / f"metrics_{s}.json"))
def app():
    t = pd.read_parquet(C.ROOT / "data/synthetic/application_train.parquet")  # use data/raw + D.read_table for real data
    return F.engineer_application(D.clean_application(t)), t


# 07 - PSI and monitoring

In [2]:
m = R('A'); print('score PSI valid->test', m['stability']['score_psi_valid_vs_test'])
pd.DataFrame(m['stability']['feature_psi_train_vs_test'])

score PSI valid->test 0.002390961630806051


,feature,psi,status
0,EXT_SOURCE_WEIGHTED,0.003499,stable
1,ANNUITY_INCOME_RATIO,0.003320,stable
2,YEARS_EMPLOYED,0.002726,stable
3,DAYS_EMPLOYED,0.002709,stable
4,CREDIT_INCOME_RATIO,0.002648,stable
5,EXT_SOURCE_MEAN,0.002207,stable
6,EXT_SOURCE_2,0.002062,stable
7,EXT_SOURCE_MIN,0.001870,stable
8,BUREAU_ENQUIRIES_TOTAL,0.001464,stable
9,AMT_REQ_CREDIT_BUREAU_YEAR,0.001370,stable


In [3]:
s = m['stability']['shift_exercise_young_vs_all']; print('shifted population (under 35): score PSI', s['score_psi']); pd.DataFrame(s['features'])

shifted population (under 35): score PSI 0.004562523256985901


,feature,psi,status
0,DAYS_EMPLOYED,0.031683,stable
1,YEARS_EMPLOYED,0.031291,stable
2,EXT_SOURCE_MEAN,0.009856,stable
3,EXT_SOURCE_MIN,0.008888,stable
4,CREDIT_INCOME_RATIO,0.006500,stable
5,EXT_SOURCE_2,0.005511,stable
6,EXT_SOURCE_WEIGHTED,0.004721,stable
7,ANNUITY_INCOME_RATIO,0.004612,stable
8,BUREAU_ENQUIRIES_TOTAL,0.003250,stable
9,AMT_REQ_CREDIT_BUREAU_YEAR,0.001284,stable


In [4]:
ST.monitoring_plan()

,What to track,Frequency,Alert threshold,Owner,Action / retrain trigger
0,Score PSI (vs. training),Monthly,<0.10 ok; 0.10-0.25 investigate; >0.25 escalate,Model owner,PSI > 0.25 two months running -> retrain review
1,Feature PSI (top 10 by SHAP),Monthly,Same thresholds,Model owner,"Check upstream data/pipeline first, then popul..."
2,Approval rate,Weekly,+/- 3 pts vs. 3-month average,Credit risk team,"Check cut-off, channel mix, data feeds"
3,Bad rate by origination cohort,Monthly (labels mature in 6-12 months),Observed > predicted by 20% relative,Risk analytics,Re-calibrate; consider retrain
4,Calibration (predicted vs observed PD by decile),Quarterly,ECE > 0.02 or Brier up > 5% relative,Model owner,Refit calibrator on recent matured cohorts
5,Ranking power (AUC / KS on matured cohorts),Quarterly,AUC drop > 0.02 vs. development,Model validation,"Retrain challenger, champion-challenger test"
6,Fairness (approval & error rates by group),Quarterly,Disparate-impact ratio < 0.8 or gap widening,Compliance + risk,"Review features/proxies, document decision"
7,"Data quality (missing rates, sentinel counts)",Daily,Missing rate change > 5 pts,Data engineering,Block scoring or fall back to manual review
